# 🚀 Qwen3 on Colab — Expose as an OpenAI-Compatible API for OpenCode

This runs Qwen3 **on Colab's GPU** using vLLM's built-in OpenAI-compatible server, then tunnels it out to a public HTTPS URL you can point OpenCode at from your local machine.

> **Requirements**: Colab with a **T4 GPU** runtime (free tier works). `Runtime → Change runtime type → T4 GPU`.

**Flow:**
1. Install vLLM + cloudflared
2. Start the vLLM OpenAI-compatible server (with an API key you generate)
3. Open a Cloudflare quick tunnel to get a public URL
4. Copy the printed `opencode.json` config into your local OpenCode setup
5. (Optional) test the endpoint from this notebook before leaving Colab
6. Stop everything when done

⚠️ Free Colab GPU sessions are time-limited and disconnect on idle — this is fine for a coding session, but the URL changes every time you restart the tunnel.

In [ ]:
# Cell 1 — Install Dependencies
!pip install -q vllm

# Install the cloudflared binary (no account/signup needed for a quick tunnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!mv cloudflared /usr/local/bin/cloudflared
print("✅ Dependencies installed.")

In [ ]:
# Cell 2 — Start the vLLM OpenAI-compatible server
import subprocess, time, secrets, requests

# --- Config ---
MODEL_ID = "Qwen/Qwen3-1.7B"   # swap for "Qwen/Qwen3-4B" or "Qwen/Qwen3-8B" for noticeably better coding quality (both still fit a T4)
PORT = 8000
MAX_MODEL_LEN = 32768           # context window; lower this if you hit GPU memory errors

API_KEY = secrets.token_urlsafe(24)  # generated locally — this is the key you'll put in OpenCode

vllm_log = open("vllm.log", "w")
vllm_proc = subprocess.Popen(
    [
        "vllm", "serve", MODEL_ID,
        "--host", "0.0.0.0",
        "--port", str(PORT),
        "--api-key", API_KEY,
        "--reasoning-parser", "qwen3",   # splits <think> content into a separate `reasoning_content` field
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", "0.85",
        "--dtype", "auto",
    ],
    stdout=vllm_log, stderr=subprocess.STDOUT,
)

print(f"⏳ Starting vLLM with {MODEL_ID} (downloads weights on first run, can take a few minutes)...")
for _ in range(180):
    try:
        if requests.get(f"http://localhost:{PORT}/health", timeout=2).status_code == 200:
            print("✅ vLLM server is up and healthy.")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(5)
else:
    print("⚠️ Server did not become healthy in time. Check vllm.log:")
    !tail -n 40 vllm.log

In [ ]:
# Cell 3 — Open a public tunnel to the server
import subprocess, time, re

tunnel_log_path = "cloudflared.log"
tunnel_log = open(tunnel_log_path, "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

PUBLIC_URL = None
print("⏳ Waiting for tunnel URL...")
for _ in range(30):
    time.sleep(2)
    with open(tunnel_log_path) as f:
        content = f.read()
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", content)
    if match:
        PUBLIC_URL = match.group(0)
        break

if PUBLIC_URL:
    print(f"✅ Public URL: {PUBLIC_URL}")
else:
    print("⚠️ Couldn't find the tunnel URL yet. Check cloudflared.log:")
    !tail -n 40 cloudflared.log

In [ ]:
# Cell 4 — Print your OpenCode config
config_snippet = f'''{{
  "$schema": "https://opencode.ai/config.json",
  "provider": {{
    "colab-qwen3": {{
      "npm": "@ai-sdk/openai-compatible",
      "name": "Colab Qwen3",
      "options": {{
        "baseURL": "{PUBLIC_URL}/v1",
        "apiKey": "{API_KEY}"
      }},
      "models": {{
        "{MODEL_ID}": {{
          "name": "{MODEL_ID} (Colab)",
          "limit": {{ "context": {MAX_MODEL_LEN}, "output": 8192 }}
        }}
      }}
    }}
  }},
  "model": "colab-qwen3/{MODEL_ID}"
}}'''

print("Add this to your local ~/.config/opencode/opencode.json (or your project's opencode.json):\n")
print(config_snippet)
print(f"\n💡 Base URL: {PUBLIC_URL}/v1")
print(f"💡 API key:  {API_KEY}")
print("\nKeep this Colab tab open and running — closing it or letting the runtime idle-disconnect will kill the server and invalidate this URL.")

In [ ]:
# Cell 5 — (Optional) test the public endpoint before switching to OpenCode
from openai import OpenAI

test_client = OpenAI(api_key=API_KEY, base_url=f"{PUBLIC_URL}/v1")

resp = test_client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "Reply with a short haiku about GPUs."}],
)
print(resp.choices[0].message.content)

---
## ⛔ Stop the server
Run the cell below to stop vLLM and close the tunnel, freeing the GPU.

In [ ]:
# Cell 6 — Stop everything
tunnel_proc.terminate()
vllm_proc.terminate()
print("🛑 Tunnel and vLLM server stopped.")
print("💡 To restart, re-run Cells 2–4 above (you'll get a new URL and API key).")